In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    datediff,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
from user_defined_functions import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_exposure_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        # "ExposureId string, SourceSystem string, PortfolioKey string, PortfolioName string,"
        # "IngestionTimestamp timestamp, BatchId string, ExposureDebtorKey string, CaseCode int,"
        # "ExposureSourceSystemKey string, AgreementKey string, PartyKey string, ExposureKey string,"
        # "PartyTaxRegistrationNumber string, CDI string, PartyName string, PartyRelationshipTypeCode string,"
        # "PartyRelationshipType string, PrimaryPartyFlag boolean, AllBorrowersDeceasedFlag boolean,"
        # "ProductKey string, ProductCode int, ExposureTypeCode string, ExposureType string,"
        # "ExposureStatusCode string, ExposureStatus string, ExposureStatusdate date, Opendate date,"
        # "ActiveExpiryDate date, Maturitydate date, OriginBank string, OriginationAmount decimal(19,2),"
        # "Currency string, BankingUnitCode int, BankingUnit string, MechanismCode string, Mechanism string,"
        # "BranchCode string, Branch string, PurposeReasonCode int, PurposeReason string,"
        # "PurposeKTCode int, PurposeKT string, MaterialityStartDate timestamp,"
        # "DaysPastDueWithMateriality decimal(15,0), DaysPastDue int, DaysPastDuePrincipal int,"
        # "DaysPastDueinterest string, BucketCode string, Bucket string, FirstArreardate date,"
        # "ExpectedOverduedate date, Overduedate date, Defaultdate date,  Denounceddate date,"
        # "MasterExposureKey string, SlaveFlag boolean,"
        # "InstalmentFrequency int, InstallmentFixedAmount decimal(19,2), NumberOfInstallments int,"
        # "FirstInstallmentDate date, NextInstallmentdate date, NextInstallmentAmount decimal(19,2),"
        # "PreviousInstallmentdate date, LastBillingdate date, NumberOfUnpaidBills string,"
        # "LastRestructuringTypeCode int, LastRestructuringType string, LastRestructuringDate date, RestructuringPEEdate date,"
        # "RestructuringPEETypeCode int, RestructuringPEEType string, ObjectiveStatusCode string,"
        # "ObjectiveStatus string, ObjectiveStatusdate date, GracePeriodEnddate date,"
        # "GracePeriodTypeCode string, GracePeriodType string, GracePeriodinterestPayableFlag string,"
        # "GracePeriodNextinterestdate timestamp, GracePeriodinterestFrequency decimal(3,0),"
        # "OffBalance decimal(19,2), OnBalance decimal(19,2), OutstandingTotalBalance string,"
        # "Principal decimal(19,2), interest decimal(19,2), Unrealizedinterest decimal(19,2),"
        # "Commission  decimal(19,2), Expense decimal(19,2), Accrued decimal(19,2),"
        # "DueAmount decimal(26,4), DuePrincipal decimal (19,2), Dueinterest decimal(19,2),"
        # "DueUnrealizedinterest decimal(19,2), DueCommission decimal (19,2), DueExpense decimal(19,2),"
        # "DueAccrued decimal(19,2), Penaltyinterest decimal(19,2),PenaltyUnrealizedinterest decimal(19,2),"
        # "InterestRate decimal(19,6), InterestCode int, InterestDescription string,"
        # "BaseRate decimal(19,6), BaseCode int, BaseDescription string,"
        # "N128Rate decimal(19,6), SpreadRate decimal(19,6), PenaltyRate decimal(19,6),"
        # "InterestRateTypeCode string, InterestRateType string, SpreadCurrencyCode string,"
        # "SpreadCurrency string, SpreadPeriod string, InterestDuration int, DayCounts int,"
        # "InterestFrequency string, InterestFixedDate date, interestValiditydate date,"
        # "InterestChangeStartdate date, InterestChangeEnddate date, InterestRateUpdatedate date,"
        # "InterestCalcStopFlag string, AutoRenewalFlag string, IsActiveFlag boolean,WriteOffAmount decimal(19,4),"
        # "WriteOffDate date,AssetSegmentCode string, AssetSegment string, DiasNumber string,"
        # "ProvisionFrozenInterestAmount string, AccountNticeBUKey string, SayFlag string,"
        # "InsuranceCompany string, PhysicalNumber string, PreCourtDate string, CourtDate string,"
        # "ContractSigningDate string, ExposureLimitAmount string, BaloonAmount decimal(15,2),"
        # "AdministrationStatusCode integer, AdministrationStatus integer, Tenor string,"
        # "IsSoldFlag boolean, IsSubsidyFlag string, IsGuaranteeFlag boolean, EverDenouncedFlag string,"
        # "EverDenouncedDate string, SPVCode string, SPV string, TotalOverdueInsterest decimal(4,0),"
        # "TotalOverdueInstallments decimal(4,0), OutstandingTotalBalanceMLT decimal(19,4),"
        # "TreatmentBranchCode int, TreatmentBranch string, LetterStatusCode string, LetterStatus string,"
        # "LetterStatusDate date, CoCStatusCode string, CoCStatus string, CoCAccountStatusDate date,"
        # "FirstLetterDate date, SecondLetterDate date, ThirdLetterDate date,"
        # "FirstLetterReminderDate date, FirstLetterFlag boolean, FirstLetterReminderFlag boolean,"
        # "SecondLetterFlag boolean, ThirdLetterFlag boolean, ExclusionDate date, ExclusionReasonCode string,"
        # "ExclusionReason string, LegalActionPriority string, DueDebtAmount decimal(19,4),url_nrm_int_amn decimal(15,2), BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Exposure"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "profits",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        #================== loan_details ==================
        loan_details_extra_filters = None

        if loan_details_extra_filters is not None:
            loan_details_extra_filters.append(date_filter)
            loan_details_final_filters = " AND ".join(loan_details_extra_filters)
        else:
            loan_details_final_filters = date_filter

        loan_details_selected_fields = [
            "eom_date",
            "prft_acc_type",
            "prft_acc_sn",
            "ACC_EXP_DT",
            "ACC_LIMIT_AMN",
            "acc_mechanism",
            "ACC_OPEN_DT",
            "acc_status",
            "ACCOUNT_NUMBER",
            "agreement_number",
            "ADJUSTMENT_DT",
            "ADJUSTMENT_GOVT_DT",
            "CASE_BILLDATE",
            "CUR_FX_INT_IND",
            "currency",
            "DRAWDOWN_EXP_DT",
            "DRAWDOWN_FST_AMN",
            "gd_adjus",
            "gd_bsect",
            "gd_lpucb",
            "gd_lpurp",
            "INSTALL_COUNT",
            "INSTALL_FIXED_AMN",
            "INSTALL_FREQ",
            "INSTALL_NEXT_DT",
            "INSTALL_PREV_DT",
            "INT_CALC_STOP_FLG",
            "INTER_CAPITALIZE_F",
            "LOAN_STATUS",
            "MORAT_INT_PAY_FLG",
            "MORATORIUM_INT_DT",
            "MORATORIUM_INT_FRQ",
            "OV_EXP_DT",
            "prft_product_code",
            "prft_acc_unit",
            "RQ_default_date",
            "ACTV_RQ_MIN_EXPIRE_DT",
            "REQ_CAP_EXPIRE_DT_OLDER",
            "gd_ADJ02",
            "ov_int_cnt",
            "ov_inst_cnt",
            "INSTALL_FIRST_DT",
            "businessdate",
        ]

        loan_details = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_details",
            loan_details_final_filters,
            loan_details_selected_fields,
        )

        loan_details = loan_details.withColumn(
            "ACCOUNT_NUMBER_Without_Prefix",
            substring(loan_details["ACCOUNT_NUMBER"], 4, 50),
        )
        # ================== loan_account_extrait ==================

        loan_account_extrait_extra_filters = None
        if loan_account_extrait_extra_filters is not None:
            loan_account_extrait_extra_filters.append(date_filter)
            loan_account_extrait_final_filters = " AND ".join(
                loan_account_extrait_extra_filters
            )
        else:
            loan_account_extrait_final_filters = date_filter
        loan_account_extrait_selected_fields = [
            "ACC_UNIT",
            "ACC_TYPE",
            "ACC_SN",
            "JUSTIFICATION_CODE",
            "EXTRAIT_COMMENTS","URL_NRM_INT_AMN",
            "businessdate",
        ]
        loan_account_extrait = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_account_extrait",
            loan_account_extrait_final_filters,
            loan_account_extrait_selected_fields,
        )

        loan_account_extrait = loan_account_extrait.distinct()  # In order to get unique values

        # ================== loan_balance ==================

        loan_balance_extra_filters = None

        if loan_balance_extra_filters is not None:
            loan_balance_extra_filters.append(date_filter)
            loan_balance_final_filters = " AND ".join(loan_balance_extra_filters)
        else:
            loan_balance_final_filters = date_filter

        loan_balance_selected_fields = [
            "EOM_DATE",
            "ACCOUNT_NUMBER",
            "NRM_ACCRUAL_AMN",
            "NRM_CAP_BAL",
            "NRM_COM_BAL",
            "NRM_EXP_BAL",
            "NRM_RL_INT_BAL",
            "NRM_URL_INT_BAL",
            "OV_ACCRUAL_AMN",
            "OV_CAP_BAL",
            "OV_COM_BAL",
            "OV_EXP_BAL",
            "OV_EXP_DT",
            "OV_RL_NRM_INT_BAL",
            "OV_RL_PNL_INT_BAL",
            "OV_URL_NRM_INT_BAL",
            "OV_URL_PNL_INT_BAL",
            "overdue_days",
            "TOTAL_BALANCE",
            "businessdate",
        ]

        loan_balance = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_balance",
            loan_balance_final_filters,
            loan_balance_selected_fields,
        )

        # ================== loan_interest ==================

        loan_interest_extra_filters = None

        if loan_interest_extra_filters is not None:
            loan_interest_extra_filters.append(date_filter)
            loan_interest_final_filters = " AND ".join(loan_interest_extra_filters)
        else:
            loan_interest_final_filters = date_filter

        loan_interest_selected_fields = [
            "ACCOUNT_NUMBER",
            "base_rate",
            "base_code",
            "base_description",
            "basedays",
            "cur_fx_int_exp_dt",
            "dur_unit",
            "duration",
            "interest_code",
            "interest_description",
            "Interest_Type",
            "n128_rate",
            "Penalty_rate",
            "spread",
            "validity_date",
            "businessdate",
        ]

        loan_interest = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_interest",
            loan_interest_final_filters,
            loan_interest_selected_fields,
        )
        loan_interest = loan_interest.distinct()  # In order to get unique values

        # ================== agreement ==================

        agreement_extra_filters = None

        if agreement_extra_filters is not None:
            agreement_extra_filters.append(date_filter)
            agreement_final_filters = " AND ".join(agreement_extra_filters)
        else:
            agreement_final_filters = date_filter

        agreement_selected_fields = ["agreement_number", "businessdate"]

        agreement = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "profits_agreement",
            agreement_final_filters,
            agreement_selected_fields,
        )

        # ================== agreement_relation ==================

        agreement_relation_extra_filters = None

        if agreement_relation_extra_filters is not None:
            agreement_relation_extra_filters.append(date_filter)
            agreement_relation_final_filters = " AND ".join(
                agreement_relation_extra_filters
            )
        else:
            agreement_relation_final_filters = date_filter

        agreement_relation_selected_fields = [
            "agreement_number",
            "CUST_ID",
            "Relation_Type",
            "MAIN_BENEF_FLG",
            "businessdate",
        ]

        agreement_relation = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "profits_agreement_relation",
            agreement_relation_final_filters,
            agreement_relation_selected_fields,
        )

        agreement_relation = (
            agreement_relation.withColumn(
                "PartyRelationshipTypeCode",
                when(
                    agreement_relation.Relation_Type.isNotNull(),
                    agreement_relation.Relation_Type,
                ),
            )
            .withColumn(
                "PartyRelationshipType",
                when(
                    (agreement_relation.MAIN_BENEF_FLG == 1)
                    & (agreement_relation.Relation_Type == "B"),
                    "Primary Owner",
                )
                .when(agreement_relation.Relation_Type == "B", "Co-Owner")
                .when(agreement_relation.Relation_Type == "G", "Guarantor"),
            )
            .distinct()
        )

        # ================== loan_relation ==================

        loan_relation_extra_filters = None

        if loan_relation_extra_filters is not None:
            loan_relation_extra_filters.append(date_filter)
            loan_relation_final_filters = " AND ".join(loan_relation_extra_filters)
        else:
            loan_relation_final_filters = date_filter

        loan_relation_selected_fields = [
            "FK_CUSTOMER",
            "BEN_G_FLAG",
            "CONTRACT",
            "FK_ACCOUNT",
        ]

        loan_relation = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_relation",
            loan_relation_final_filters,
            loan_relation_selected_fields,
        )
        loan_relation = (
            loan_relation.withColumn("CONTRACT", trim(col("CONTRACT")))
            .withColumn(
                "PartyRelationshipTypeCode",
                when(loan_relation.BEN_G_FLAG.isin([0, 1]), lit("B")).when(
                    loan_relation.BEN_G_FLAG == 3, lit("G")
                ),
            )
            .withColumn(
                "PartyRelationshipType",
                when(loan_relation.BEN_G_FLAG.isin([0, 1]), "Co-Owner").when(
                    loan_relation.BEN_G_FLAG == 3, "Guarantor"
                ),
            )
        )

        # ================== customer ==================

        customer_extra_filters = None

        if customer_extra_filters is not None:
            customer_extra_filters.append(date_filter)
            customer_final_filters = " AND ".join(customer_extra_filters)
        else:
            customer_final_filters = date_filter

        customer_selected_fields = [
            "CUST_ID",
            "CDI",
            "tax_number",
            "NAME_STANDARD",
            "businessdate",
        ]

        customer = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "profits_customer",
            customer_final_filters,
            customer_selected_fields,
        )

        # ================== loan_add_info ==================

        loan_add_info_extra_filters = None

        if loan_add_info_extra_filters is not None:
            loan_add_info_extra_filters.append(date_filter)
            loan_add_info_final_filters = " AND ".join(loan_add_info_extra_filters)
        else:
            loan_add_info_final_filters = date_filter

        loan_add_info_selected_fields = [
            "AMOUNT_DATA",
            "ACC_SN",
            "ACC_UNIT",
            "ACC_TYPE",
            "ROW_ID",
            "NUM_DATA",
            "DATE_DATA",
        ]

        loan_add_info = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_add_info",
            loan_add_info_final_filters,
            loan_add_info_selected_fields,
        )

        # ========================== loan_product ==========================

        loan_product_extra_filters = None

        if loan_product_extra_filters is not None:
            loan_product_extra_filters.append(date_filter)
            loan_product_final_filters = " AND ".join(loan_product_extra_filters)

        else:
            loan_product_final_filters = date_filter

        loan_product_selected_fields = [
            "product_code",
        ]

        loan_product = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_product",
            loan_product_final_filters,
            loan_product_selected_fields,
        )

        # ================== customer_RF_codes ==================

        customer_RF_codes_extra_filters = None

        if customer_RF_codes_extra_filters is not None:
            customer_RF_codes_extra_filters.append(date_filter)
            customer_RF_codes_final_filters = " AND ".join(
                customer_RF_codes_extra_filters
            )
        else:
            customer_RF_codes_final_filters = date_filter

        customer_RF_codes_selected_fields = [
            "cust_rf_id",
            "customer_code",
            "ACCOUNT_NUMBER",
            "ACCOUNT_SYSTEM",
        ]

        customer_RF_codes = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "profits_customer_RF_codes",
            customer_RF_codes_final_filters,
            customer_RF_codes_selected_fields,
        )
        customer_RF_codes = customer_RF_codes.distinct()

        # ================== params ==================

        params_extra_filters = None

        if params_extra_filters is not None:
            params_extra_filters.append(date_filter)
            params_final_filters = " AND ".join(params_extra_filters)
        else:
            params_final_filters = date_filter

        params_selected_fields = [
            "param_type",
            "serial_num",
            "description",
            "param_txt_1",
            "businessdate",
        ]

        params = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "profits_params",
            params_final_filters,
            params_selected_fields,
        )
        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = ["PortfolioName", "Prefix", "unit_code_profits", "originBank"]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        # ================== user_defined_fields ==================

        user_defined_fields_extra_filters = None
        if user_defined_fields_extra_filters is not None:
            user_defined_fields_extra_filters.append(date_filter)
            user_defined_fields_final_filters = " AND ".join(
                user_defined_fields_extra_filters
            )
        else:
            user_defined_fields_final_filters = date_filter
        user_defined_fields_selected_fields = [
            "UD_CUSTOMER_CODE",
            "UD_ACCOUNT_NUMBER",
            "PFG_TAG_SET_CODE",
            "PFG_TAG",
            "FIELD_LABEL",
            "FIELD_VALUE",
            "businessdate",
        ]
        user_defined_fields = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_user_defined_fields",
            user_defined_fields_final_filters,
            user_defined_fields_selected_fields,
        )

        # =================== agreement_relation_new ==================
        agreement_relation_new = agreement_relation.join(
            loan_details,
            agreement_relation["agreement_number"] == loan_details["agreement_number"],
            "inner",
        )

        agreement_relation_new = agreement_relation_new.select(
            loan_details.agreement_number,
            agreement_relation_new.ACCOUNT_NUMBER,
            agreement_relation_new.CUST_ID,
            agreement_relation.PartyRelationshipTypeCode,
            agreement_relation.PartyRelationshipType,
            agreement_relation.MAIN_BENEF_FLG,
        )

        # ================= loan_relation_new ==========================
        loan_relation_new = (
            loan_relation.join(
                loan_details,
                loan_relation["FK_ACCOUNT"] == loan_details["ACCOUNT_NUMBER"],
                "inner",
            )
            .join(
                agreement_relation,
                (
                    agreement_relation["agreement_number"]
                    == loan_details["agreement_number"]
                )
                & (agreement_relation["cust_id"] == loan_relation["FK_CUSTOMER"]),
                "left",
            )
            .filter(
                agreement_relation["agreement_number"].isNull(),
            )
        )

        # ========================== profits_account ==========================

        profits_account_extra_filters = None

        if profits_account_extra_filters is not None:
            profits_account_extra_filters.append(date_filter)
            profits_account_final_filters = " AND ".join(profits_account_extra_filters)

        else:
            profits_account_final_filters = date_filter

        profits_account_selected_fields = [
            "LNS_OPEN_UNIT",
            "ACCOUNT_NUMBER",
            "LNS_OPEN_UNIT",
            "LNS_SN",
            "LNS_TYPE",
            "PRFT_SYSTEM",
        ]

        profits_account = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_profits_account",
            profits_account_final_filters,
            profits_account_selected_fields,
        )

        loan_relation_new = loan_relation_new.select(
            loan_details.agreement_number,
            loan_relation_new.FK_ACCOUNT.alias("ACCOUNT_NUMBER"),
            loan_relation_new.FK_CUSTOMER.alias("CUST_ID"),
            loan_relation.PartyRelationshipTypeCode,
            loan_relation.PartyRelationshipType,
            lit(None).alias("MAIN_BENEF_FLG"),
        )

        # ================== loan_relation & agreement relation Master Table ==============

        loan_relation_agreement_relation = loan_relation_new.union(
            agreement_relation_new
        )
        loan_relation_agreement_relation = (
            loan_relation_agreement_relation.withColumnRenamed(
                "PartyRelationshipTypeCode", "PartyRelationshipTypeCodeF"
            )
            .withColumnRenamed("PartyRelationshipType", "PartyRelationshipTypeF")
            .withColumnRenamed("agreement_number", "agreement_numberF")
            .withColumnRenamed("ACCOUNT_NUMBER", "ACCOUNT_NUMBERF")
            .withColumnRenamed("CUST_ID", "CUST_IDF")
        )

        # ================== user_defined_fields_G64 ==================
        user_defined_fields_G64 = (
            user_defined_fields.where(
                (user_defined_fields["PFG_TAG_SET_CODE"] == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
                & (user_defined_fields["PFG_TAG"] == "PFTAG64")
            )
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_G64")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_G64")
        )

        user_defined_fields_G65 = (
            user_defined_fields.where(
                (user_defined_fields["PFG_TAG_SET_CODE"] == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
                & (user_defined_fields["PFG_TAG"] == "PFTAG65")
                & (user_defined_fields["FIELD_VALUE"] == "1")
            )
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_G65")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_G65")
        )

        user_defined_fields_G66 = (
            user_defined_fields.where(
                (user_defined_fields["PFG_TAG_SET_CODE"] == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
                & (user_defined_fields["PFG_TAG"] == "PFTAG66")
                & (user_defined_fields["FIELD_VALUE"] == "2")
            )
            .withColumnRenamed("UD_ACCOUNT_NUMBER", "UD_ACCOUNT_NUMBER_G66")
            .withColumnRenamed("PFG_TAG_SET_CODE", "PFG_TAG_SET_CODE_G66")
        )

        user_defined_fields_G64_G65_join_condition = (
            user_defined_fields_G64["UD_ACCOUNT_NUMBER_G64"]
            == user_defined_fields_G65["UD_ACCOUNT_NUMBER_G65"]
        ) & (
            user_defined_fields_G64["PFG_TAG_SET_CODE_G64"]
            == user_defined_fields_G65["PFG_TAG_SET_CODE_G65"]
        )

        user_defined_fields_G64_G66_join_condition = (
            user_defined_fields_G64["UD_ACCOUNT_NUMBER_G64"]
            == user_defined_fields_G66["UD_ACCOUNT_NUMBER_G66"]
        ) & (
            user_defined_fields_G64["PFG_TAG_SET_CODE_G64"]
            == user_defined_fields_G66["PFG_TAG_SET_CODE_G66"]
        )


        # =================== M1_df_1 ==================
        M1_df_1 = (user_defined_fields.where(
                    (user_defined_fields["PFG_TAG_SET_CODE"] == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
                    & (user_defined_fields["FIELD_LABEL"].like("ΤΥΠΟΣ ΛΟΓΑΡΙΑΣΜΟΥ%"))
                    & (user_defined_fields["FIELD_VALUE"] == "2")
                )
                .withColumnRenamed("UD_ACCOUNT_NUMBER", "MASTER_ACC")
                .select("MASTER_ACC")
            )

        # =================== M2_df_1 ==================
        M2_df_1 = (user_defined_fields.where(
                    (user_defined_fields["PFG_TAG_SET_CODE"] == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
                    & (user_defined_fields["FIELD_LABEL"].like("ΤΥΠΟΣ ΣΥΝΔΕΣΗΣ%"))
                    & (user_defined_fields["FIELD_VALUE"].isin("2", "3", "4", "5", "6", "7"))
                )
                .withColumnRenamed("UD_ACCOUNT_NUMBER", "SLAVE_ACC")
                .select("SLAVE_ACC")
            )

        # =================== M1_df_2 ==================
        M1_df_2 = (user_defined_fields.where(
                    (user_defined_fields["PFG_TAG_SET_CODE"] == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
                    & (user_defined_fields["FIELD_LABEL"].like("ΤΥΠΟΣ ΛΟΓΑΡΙΑΣΜΟΥ%"))
                    & (user_defined_fields["FIELD_VALUE"] == "6")
                )
                .withColumnRenamed("UD_ACCOUNT_NUMBER", "MASTER_ACC")
                .select("MASTER_ACC")
            )

        # =================== M2_df_2 ==================
        M2_df_2 = (user_defined_fields.where(
                    (user_defined_fields["PFG_TAG_SET_CODE"] == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
                    & (user_defined_fields["FIELD_LABEL"].like("ΤΥΠΟΣ ΣΥΝΔΕΣΗΣ%"))
                    & (user_defined_fields["FIELD_VALUE"] == "8")
                )
                .withColumnRenamed("UD_ACCOUNT_NUMBER", "SLAVE_ACC")
                .select("SLAVE_ACC")
            )

        # =================== M1_df_3 ==================
        M1_df_3 = (user_defined_fields.where(
                    (user_defined_fields["PFG_TAG_SET_CODE"] == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
                    & (user_defined_fields["FIELD_LABEL"].like("ΤΥΠΟΣ ΛΟΓΑΡΙΑΣΜΟΥ%"))
                    & (user_defined_fields["FIELD_VALUE"] == "3")
                )
                .withColumnRenamed("UD_ACCOUNT_NUMBER", "MASTER_ACC")
                .select("MASTER_ACC")
            )

        # =================== M2_df_3 ==================
        M2_df_3 = (user_defined_fields.where(
                    (user_defined_fields["PFG_TAG_SET_CODE"] == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
                    & (user_defined_fields["FIELD_LABEL"].like("ΤΥΠΟΣ ΣΥΝΔΕΣΗΣ%"))
                    & (user_defined_fields["FIELD_VALUE"] == "4")
                )
                .withColumnRenamed("UD_ACCOUNT_NUMBER", "SLAVE_ACC")
                .select("SLAVE_ACC")
        )

        loan_details_Agreement_join_condition = (
            loan_details["Agreement_Number"] == agreement["Agreement_Number"]
        )

        loan_details_Loan_Relation_Agreement_Relation_join_condition = (
            loan_details["ACCOUNT_NUMBER"]
            == loan_relation_agreement_relation["ACCOUNT_NUMBERF"]
        )

        Loan_Relation_Agreement_Relation_customer_join_condition = (
            loan_relation_agreement_relation["cust_idF"] == customer["cust_id"]
        )

        loan_details_Loan_balance_join_condition = (
            loan_details["ACCOUNT_NUMBER"] == loan_balance["ACCOUNT_NUMBER"]
        )

        loan_details_loan_interest_join_condition = (
            loan_details["ACCOUNT_NUMBER"] == loan_interest["ACCOUNT_NUMBER"]
        )

        loan_product_loan_details_join_condition = (
            loan_product["product_code"] == loan_details["prft_product_code"]
        )

        user_defined_fields_loan_details_join_condition = (
            trim(user_defined_fields["UD_ACCOUNT_NUMBER"]) == trim(loan_details["ACCOUNT_NUMBER"])
        )

        user_defined_fields_M1_df_1_join_condition = (
            trim(user_defined_fields["UD_ACCOUNT_NUMBER"]) == trim(M1_df_1["MASTER_ACC"])
        )

        user_defined_fields_M2_df_1_join_condition = (
            trim(user_defined_fields["FIELD_VALUE"]) == trim(M2_df_1["SLAVE_ACC"])
        )
        
        user_defined_fields_M1_df_2_join_condition = (
            trim(user_defined_fields["UD_ACCOUNT_NUMBER"]) == trim(M1_df_2["MASTER_ACC"])
        )

        user_defined_fields_M2_df_2_join_condition = (
            trim(user_defined_fields["FIELD_VALUE"]) == trim(M2_df_2["SLAVE_ACC"])
        )
        
        user_defined_fields_M1_df_3_join_condition = (
            trim(user_defined_fields["UD_ACCOUNT_NUMBER"]) == trim(M1_df_3["MASTER_ACC"])
        )

        user_defined_fields_M2_df_3_join_condition = (
            trim(user_defined_fields["FIELD_VALUE"]) == trim(M2_df_3["SLAVE_ACC"])
        )

        # ================== loan_add_info_profits_account_join ==================

        loan_add_info_profits_account_join_condition = (
            (profits_account["LNS_OPEN_UNIT"] == loan_add_info["ACC_UNIT"])
            & (profits_account["LNS_TYPE"] == loan_add_info["ACC_TYPE"])
            & (profits_account["LNS_SN"] == loan_add_info["ACC_SN"])
            & (profits_account["PRFT_SYSTEM"] == 4)
            & (loan_add_info["ROW_ID"] == 84)
        )

        loan_add_info_profits_account = loan_add_info.join(
            profits_account, loan_add_info_profits_account_join_condition, "inner"
        )

        loan_add_info_profits_account = loan_add_info_profits_account.select(
            profits_account.ACCOUNT_NUMBER.cast("string"),
            loan_add_info.NUM_DATA.alias("NUM_DATA"),
            loan_add_info.DATE_DATA.alias("DATE_DATA"),
        )

        loan_add_info_profits_account_loan_details_join_condition = (
            loan_add_info_profits_account["ACCOUNT_NUMBER"]
            == loan_details["ACCOUNT_NUMBER"]
        )

        # ================== params description tables ==================

        params_loan_status = params.where(
            params["param_txt_1"] == "LOAN STATUS LIST"
        ).withColumnRenamed("description", "params_loan_status_description")

        loan_details_params_loan_status_join_condition = (
            loan_details["LOAN_STATUS"] == params_loan_status["serial_num"]
        )

        params_purpose_reason = params.where(
            params["param_type"] == "LPURP"
        ).withColumnRenamed("description", "params_purpose_reason_description")

        loan_details_params_purpose_reason_join_condition = (
            loan_details["gd_lpurp"] == params_purpose_reason["serial_num"]
        )

        params_purposeKT = params.where(
            params["param_type"] == "LPUCB"
        ).withColumnRenamed("description", "params_purposeKT_description")

        loan_details_params_purposeKT_join_condition = (
            loan_details["gd_lpucb"] == params_purposeKT["serial_num"]
        )

        params_RestructuringType = params.where(
            params["param_type"] == "ADJUS"
        ).withColumnRenamed("description", "params_RestructuringType_description")

        loan_details_params_RestructuringType_join_condition = (
            loan_details["gd_adjus"] == params_RestructuringType["serial_num"]
        )

        params_RestructuringPEEType = params.where(
            params["param_type"] == "ADJ02"
        ).withColumnRenamed("description", "params_RestructuringPEEType_description")

        loan_details_params_RestructuringPEEType_join_condition = (
            loan_details["gd_ADJ02"] == params_RestructuringPEEType["serial_num"]
        )

        loan_details_final_portfolio_join_condition = (
            loan_details["prft_acc_unit"] == portfolio["unit_code_profits"]
        )

        customer_RF_codes_join_condition = (
            (loan_relation_agreement_relation["cust_idF"] == customer_RF_codes["customer_code"])
            & (loan_details["ACCOUNT_NUMBER"] == customer_RF_codes["ACCOUNT_NUMBER"])
            & (customer_RF_codes["ACCOUNT_SYSTEM"] == 4)
        )
        
        loan_add_info_loan_details_join_condition = (
            (loan_add_info["ACC_SN"] == loan_details["prft_acc_sn"])
            & (loan_add_info["ACC_TYPE"] == loan_details["prft_acc_type"])
            & (loan_add_info["ACC_UNIT"] == loan_details["prft_acc_unit"])
            & (loan_add_info["ROW_ID"] == 83)
        )

        loan_account_extrait_loan_details_join_condition = (
            (loan_account_extrait["ACC_SN"] == loan_details["prft_acc_sn"])
            & (loan_account_extrait["ACC_TYPE"] == loan_details["prft_acc_type"])
            & (loan_account_extrait["ACC_UNIT"] == loan_details["prft_acc_unit"])
            & (loan_account_extrait["JUSTIFICATION_CODE"] == 43023)
        )

        # Joins
        slave_df_1 = (
            user_defined_fields.join(
                M1_df_1,
                user_defined_fields_M1_df_1_join_condition,
                "INNER",
            )
            .join(
                M2_df_1,
                user_defined_fields_M2_df_1_join_condition,
                "INNER",
            )
            .join(
                loan_details,
                user_defined_fields_loan_details_join_condition,
                "INNER",
            )
        )

        slave_df_2 = (
            user_defined_fields.join(
                M1_df_2,
                user_defined_fields_M1_df_2_join_condition,
                "INNER",
            )
            .join(
                M2_df_2,
                user_defined_fields_M2_df_2_join_condition,
                "INNER",
            )
            .join(
                loan_details,
                user_defined_fields_loan_details_join_condition,
                "INNER",
            )
        )

        slave_df_3 = (
            user_defined_fields.join(
                M1_df_3,
                user_defined_fields_M1_df_3_join_condition,
                "INNER",
            )
            .join(
                M2_df_3,
                user_defined_fields_M2_df_3_join_condition,
                "INNER",
            )
            .join(
                loan_details,
                user_defined_fields_loan_details_join_condition,
                "INNER",
            )
        )

        slave_df_1 = slave_df_1.filter(
            (user_defined_fields.PFG_TAG_SET_CODE == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
            & (user_defined_fields.FIELD_LABEL.like("ΑΡΙΘΜΟΣ ΛΟΓΑΡΙΑΣΜΟΥ%"))
            & (user_defined_fields.FIELD_VALUE != '')
        )

        slave_df_1 = slave_df_1.select(
            trim(user_defined_fields.UD_ACCOUNT_NUMBER).alias("MASTER_ACC"),
            trim(user_defined_fields.FIELD_VALUE).alias("SLAVE_ACC")
        )

        slave_df_2 = slave_df_2.filter(
            (user_defined_fields.PFG_TAG_SET_CODE == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
            & (user_defined_fields.FIELD_LABEL.like("ΑΡΙΘΜΟΣ ΛΟΓΑΡΙΑΣΜΟΥ%"))
            & (user_defined_fields.FIELD_VALUE != '')
        )

        slave_df_2 = slave_df_2.select(
            trim(user_defined_fields.UD_ACCOUNT_NUMBER).alias("MASTER_ACC"),
            trim(user_defined_fields.FIELD_VALUE).alias("SLAVE_ACC")
        )

        slave_df_3 = slave_df_3.filter(
            (user_defined_fields.PFG_TAG_SET_CODE == "ΣΥΝΔ. ΛΟΓΑΡΙΑΣΜΟΣ")
            & (user_defined_fields.FIELD_LABEL.like("ΑΡΙΘΜΟΣ ΛΟΓΑΡΙΑΣΜΟΥ%"))
            & (user_defined_fields.FIELD_VALUE != '')
        )

        slave_df_3 = slave_df_3.select(
            trim(user_defined_fields.UD_ACCOUNT_NUMBER).alias("MASTER_ACC"),
            trim(user_defined_fields.FIELD_VALUE).alias("SLAVE_ACC")
        )

        slave_df = (
            slave_df_1.union(slave_df_2)
                      .union(slave_df_3)
        )

        slave_df = slave_df.filter(
            (slave_df.MASTER_ACC != slave_df.SLAVE_ACC)
        ).select(slave_df.SLAVE_ACC).distinct()

        slave_df.createOrReplaceTempView("slave_df")

        target_table_profits = (
            loan_details.join(agreement, loan_details_Agreement_join_condition, "inner")
            .join(
                loan_relation_agreement_relation,
                loan_details_Loan_Relation_Agreement_Relation_join_condition,
                "inner",
            )
            .join(
                customer,
                Loan_Relation_Agreement_Relation_customer_join_condition,
                "inner",
            )
            .join(loan_balance, loan_details_Loan_balance_join_condition, "left")
            .join(loan_interest, loan_details_loan_interest_join_condition, "left")
            .join(
                params_loan_status,
                loan_details_params_loan_status_join_condition,
                "left",
            )
            .join(
                params_purpose_reason,
                loan_details_params_purpose_reason_join_condition,
                "left",
            )
            .join(
                params_purposeKT, loan_details_params_purposeKT_join_condition, "left"
            )
            .join(
                params_RestructuringType,
                loan_details_params_RestructuringType_join_condition,
                "left",
            )
            .join(
                portfolio,
                loan_details_final_portfolio_join_condition,
                "left",
            )
            .join(
                customer_RF_codes,
                customer_RF_codes_join_condition,
                "left",
            )
            .join(loan_add_info, loan_add_info_loan_details_join_condition, "left")
            .join(
                loan_add_info_profits_account,
                loan_add_info_profits_account_loan_details_join_condition,
                "left",
            )
            .join(
                params_RestructuringPEEType,
                loan_details_params_RestructuringPEEType_join_condition,
                "left",
            )
            .join(loan_product, loan_product_loan_details_join_condition, "left")
            .join(slave_df, trim(slave_df["SLAVE_ACC"]) == trim(loan_details["ACCOUNT_NUMBER"]), "left")
            .join(loan_account_extrait,
                  loan_account_extrait_loan_details_join_condition,
                  "left")
        )

        target_table_profits_ExposureId_primary_key_fields = [
            loan_details.ACCOUNT_NUMBER,
            loan_relation_agreement_relation["CUST_IDF"],
            loan_relation_agreement_relation["PartyRelationshipTypeF"],
            lit("profits"),
        ]
        target_table_profits_ExposureDebtorKey_primary_key_fields = [
            loan_details.ACCOUNT_NUMBER,
            loan_relation_agreement_relation["CUST_IDF"],
            loan_relation_agreement_relation["PartyRelationshipTypeF"],
        ]
        target_table_profits_PartyKey_primary_key_fields = [
            loan_relation_agreement_relation["CUST_IDF"]
        ]
        target_table_profits_ExposureSourceSystemKey_primary_key_fields = [
            loan_details.prft_acc_unit,
            loan_details.prft_acc_type,
            loan_details.prft_acc_sn,
        ]

        target_table_profits = (
            target_table_profits.withColumn(
                "ExposureId",
                sha2(
                    concat_ws(";", *target_table_profits_ExposureId_primary_key_fields),
                    256,
                ),
            )
            .withColumn(
                "ExposureDebtorKey",
                concat_ws(
                    ";", *target_table_profits_ExposureDebtorKey_primary_key_fields
                ),
            )
            .withColumn("PartyKey", *target_table_profits_PartyKey_primary_key_fields)
            .withColumn(
                "ExposureSourceSystemKey",
                concat_ws(
                    ";",
                    *target_table_profits_ExposureSourceSystemKey_primary_key_fields,
                ),
            )
            .withColumn(
                "DueAmount",
                coalesce(loan_balance.OV_CAP_BAL, lit(0).cast("decimal(19,2)"))
                + coalesce(loan_balance.OV_RL_NRM_INT_BAL, lit(0).cast("decimal(19,2)"))
                + coalesce(
                    loan_balance.OV_URL_NRM_INT_BAL, lit(0).cast("decimal(19,2)")
                )
                + coalesce(loan_balance.OV_RL_PNL_INT_BAL, lit(0).cast("decimal(19,2)"))
                + coalesce(
                    loan_balance.OV_URL_PNL_INT_BAL, lit(0).cast("decimal(19,2)")
                )
                + coalesce(loan_balance.OV_COM_BAL, lit(0).cast("decimal(19,2)"))
                + coalesce(loan_balance.OV_EXP_BAL, lit(0).cast("decimal(19,2)"))
                + coalesce(loan_balance.OV_ACCRUAL_AMN, lit(0).cast("decimal(19,2)")),
            )
            .withColumn(
                "mechanism",
                when(loan_details.acc_mechanism == 1, "Fixed Maturity Open Loan")
                .when(loan_details.acc_mechanism == 2, "Revolving Loan")
                .when(
                    loan_details.acc_mechanism == 3,
                    "Revolving Loan with Promisory Notes",
                )
                .when(loan_details.acc_mechanism == 4, "Amortised Loan")
                .when(loan_details.acc_mechanism == 5, "Amortised Loan (Fixed Capital)")
                .when(
                    loan_details.acc_mechanism == 6,
                    "Amortised Loan (Unequal Installments)",
                )
                .when(loan_details.acc_mechanism == 7, "Open Credit"),
            )
            .withColumn(
                "IsActiveFlag",
                when(loan_details.acc_status == 3, False).otherwise(True),
            )
            .withColumn(
                "AssetSegment",
                when(loan_details.gd_lpucb == "99500", "CL")
                .when(loan_details.gd_lpucb == "99600", "ML")
                .when(loan_details.gd_lpucb == "99700", "SB")
                .otherwise(lit(None)),
            )
            .withColumn(
                "SlaveFlag",
                when(
                    slave_df.SLAVE_ACC.isNull(), False
                ).otherwise(True)
            )
            .withColumn(
                "MasterExposureKey", lit(None)
            )
            .withColumn(
                "DaysPastDuePrincipal",
                datediff(
                    loan_details["EOM_DATE"], loan_details["REQ_CAP_EXPIRE_DT_OLDER"]
                ),
            )
            .withColumn(
                "ProductKey",
                concat_ws(
                    ";",
                    loan_product.product_code,
                    loan_details.gd_bsect,
                ),
            )
            .withColumn(
                "BucketCodeCalc",
                get_bucket_from_dpd_udf(
                     coalesce(loan_add_info_profits_account.NUM_DATA, lit(0))  #KS 17/6/2025 REPBTC-117
                ),
            )             
        )

        # ================== Selection of Final Columns ==================

        result_df_profits = target_table_profits.select(
            target_table_profits.ExposureId.alias("ExposureId"),
            lit("profits").alias("SourceSystem"),
            loan_details.prft_acc_unit.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            target_table_profits.ExposureDebtorKey.alias("ExposureDebtorKey"),
            lit(None).alias("CaseCode"),
            target_table_profits.ExposureSourceSystemKey.cast("string").alias(
                "ExposureSourceSystemKey"
            ),
            loan_details.ACCOUNT_NUMBER.cast("string").alias("ExposureKey"),
            loan_details.agreement_number.cast("string").alias("AgreementKey"),
            target_table_profits.PartyKey.cast("string").alias("PartyKey"),
            customer.CDI.alias("CDI"),
            customer.NAME_STANDARD.alias("PartyName"),
            customer.tax_number.cast("string").alias("PartyTaxRegistrationNumber"),
            loan_relation_agreement_relation.PartyRelationshipTypeCodeF.alias(
                "PartyRelationshipTypeCode"
            ),
            loan_relation_agreement_relation.PartyRelationshipTypeF.alias(
                "PartyRelationshipType"
            ),
            loan_relation_agreement_relation.MAIN_BENEF_FLG.cast("boolean").alias(
                "PrimaryPartyFlag"
            ),
            target_table_profits.ProductKey.cast("string").alias("ProductKey"),
            loan_details.prft_product_code.alias("ProductCode"),
            lit(None).alias("ExposureTypeCode"),
            lit(None).alias("ExposureType"),  # same
            loan_details.LOAN_STATUS.alias("ExposureStatusCode"),
            params_loan_status.params_loan_status_description.alias("ExposureStatus"),
            lit(None).alias("ExposureStatusDate"),
            loan_details.ACC_OPEN_DT.alias("OpenDate"),
            loan_details.ACTV_RQ_MIN_EXPIRE_DT.alias("ActiveExpiryDate"),
            loan_details.ACC_EXP_DT.alias("MaturityDate"),
            portfolio.originBank.cast("string").alias("OriginBank"),
            loan_details.DRAWDOWN_FST_AMN.alias("OriginationAmount"),
            loan_details.currency.alias("Currency"),
            loan_details.gd_bsect.alias("BankingUnitCode"),
            lit(None).alias("BankingUnit"),
            loan_details.acc_mechanism.alias("MechanismCode"),
            target_table_profits.mechanism.alias("Mechanism"),
            lit(None).alias("BranchCode"),
            lit(None).alias("Branch"),
            loan_details.gd_lpurp.cast("int").alias("PurposeReasonCode"),
            params_purpose_reason.params_purpose_reason_description.alias(
                "PurposeReason"
            ),
            loan_details.gd_lpucb.alias("PurposeKTCode"),
            params_purposeKT.params_purposeKT_description.alias("PurposeKT"),
            loan_add_info_profits_account.DATE_DATA.alias("MaterialityStartDate"),
            loan_add_info_profits_account.NUM_DATA.alias("DaysPastDueWithMateriality"),
            loan_balance.overdue_days.alias("DaysPastDue"),
            target_table_profits.DaysPastDuePrincipal.alias("DaysPastDuePrincipal"),
            lit(None).alias("DaysPastDueInterest"),
            target_table_profits.BucketCodeCalc.cast("string").alias("BucketCode"),
            target_table_profits.BucketCodeCalc.cast("string").alias("Bucket"),            
            loan_balance.OV_EXP_DT.alias("FirstArrearDate"),
            lit(None).alias("ExpectedOverdueDate"),
            loan_details.OV_EXP_DT.alias("OverdueDate"),
            loan_details.RQ_default_date.alias("DefaultDate"),
            # lit(None).alias("DenouncedStatusCode"),
            # lit(None).alias("DenouncedStatus"),
            lit(None).alias("DenouncedDate"),
            target_table_profits.MasterExposureKey.cast("string").alias(
                "MasterExposureKey"
            ),
            target_table_profits.SlaveFlag.alias("SlaveFlag"),
            loan_details.INSTALL_FREQ.alias("InstalmentFrequency"),
            loan_details.INSTALL_FIXED_AMN.alias("InstallmentFixedAmount"),
            loan_details.INSTALL_COUNT.alias("NumberOfInstallments"),
            loan_details.INSTALL_FIRST_DT.alias("FirstInstallmentDate"),
            loan_details.INSTALL_NEXT_DT.alias("NextInstallmentDate"),
            lit(None).alias("NextInstallmentAmount"),
            loan_details.INSTALL_PREV_DT.alias("PreviousInstallmentDate"),
            loan_details.CASE_BILLDATE.alias("LastBillingDate"),
            lit(None).alias("NumberOfUnpaidBills"),
            loan_details.gd_adjus.alias("LastRestructuringTypeCode"),
            params_RestructuringType.params_RestructuringType_description.alias(
                "LastRestructuringType"
            ),
            loan_details.ADJUSTMENT_DT.cast("date").alias("LastRestructuringDate"),
            loan_details.ADJUSTMENT_GOVT_DT.alias("RestructuringPEEDate"),
            loan_details.gd_ADJ02.cast("int").alias("RestructuringPEETypeCode"),
            params_RestructuringPEEType.params_RestructuringPEEType_description.cast(
                "string"
            ).alias("RestructuringPEEType"),
            lit(None).alias("ObjectiveStatusCode"),
            lit(None).alias("ObjectiveStatus"),
            lit(None).alias("ObjectiveStatusDate"),
            loan_details.DRAWDOWN_EXP_DT.alias("GracePeriodEndDate"),
            loan_details.INTER_CAPITALIZE_F.alias("GracePeriodTypeCode"),
            lit(None).alias("GracePeriodType"),
            loan_details.MORAT_INT_PAY_FLG.alias("GracePeriodInterestPayableFlag"),
            loan_details.MORATORIUM_INT_DT.alias("GracePeriodNextInterestDate"),
            loan_details.MORATORIUM_INT_FRQ.alias("GracePeriodInterestFrequency"),
            lit(None).alias("OffBalance"),
            lit(None).alias("OnBalance"),
            loan_balance.TOTAL_BALANCE.cast("string").alias("OutstandingTotalBalance"),
            loan_balance.NRM_CAP_BAL.alias("Principal"),
            loan_balance.NRM_RL_INT_BAL.alias("Interest"),
            loan_balance.NRM_URL_INT_BAL.alias("UnrealizedInterest"),
            loan_balance.NRM_COM_BAL.alias("Commission"),
            loan_balance.NRM_EXP_BAL.alias("Expense"),
            loan_balance.NRM_ACCRUAL_AMN.alias("Accrued"),
            target_table_profits.DueAmount.cast("decimal(26,4)").alias("DueAmount"),
            loan_balance.OV_CAP_BAL.alias("DuePrincipal"),
            loan_balance.OV_RL_NRM_INT_BAL.alias("DueInterest"),
            loan_balance.OV_URL_NRM_INT_BAL.alias("DueUnrealizedInterest"),
            loan_balance.OV_COM_BAL.alias("DueCommission"),
            loan_balance.OV_EXP_BAL.alias("DueExpense"),
            loan_balance.OV_ACCRUAL_AMN.alias("DueAccrued"),
            loan_balance.OV_RL_PNL_INT_BAL.alias("PenaltyInterest"),
            loan_balance.OV_URL_PNL_INT_BAL.alias("PenaltyUnrealizedInterest"),
            (loan_interest.base_rate + loan_interest.spread + loan_interest.n128_rate)
            .cast("decimal(19,6)")
            .alias("InterestRate"),
            loan_interest.interest_code.alias("InterestCode"),
            loan_interest.interest_description.alias("InterestDescription"),
            loan_interest.base_rate.cast("decimal(19,6)").alias("BaseRate"),
            loan_interest.base_code.alias("BaseCode"),
            loan_interest.base_description.alias("BaseDescription"),
            loan_interest.n128_rate.alias("N128Rate"),
            loan_interest.spread.alias("SpreadRate"),
            loan_interest.Penalty_rate.alias("PenaltyRate"),
            loan_interest.Interest_Type.alias("InterestRateTypeCode"),
            loan_interest.Interest_Type.alias("InterestRateType"),
            lit(None).alias("SpreadCurrencyCode"),
            lit(None).alias("SpreadCurrency"),
            lit(None).alias("SpreadPeriod"),
            loan_interest.duration.alias("InterestDuration"),
            loan_interest.basedays.alias("DayCounts"),
            loan_interest.dur_unit.alias("InterestFrequency"),
            loan_interest.cur_fx_int_exp_dt.alias("InterestFixedDate"),
            loan_interest.validity_date.alias("InterestValidityDate"),
            lit(None).alias("InterestChangeStartDate"),
            lit(None).alias("InterestChangeEndDate"),
            lit(None).alias("InterestRateUpdateDate"),
            loan_details.INT_CALC_STOP_FLG.alias("InterestCalcStopFlag"),
            loan_details.CUR_FX_INT_IND.alias("AutoRenewalFlag"),
            target_table_profits.IsActiveFlag.alias("IsActiveFlag"),
            lit(None).alias("WriteOffDate"),
            lit(None).alias("WriteOffAmount"),
            loan_details.gd_lpucb.cast("string").alias("AssetSegmentCode"),
            target_table_profits.AssetSegment.cast("string").alias("AssetSegment"),
            customer_RF_codes.cust_rf_id.cast("string").alias("DiasNumber"),
            lit(None).cast("string").alias("ProvisionFrozenInterestAmount"),
            lit(None).cast("string").alias("AccountNticeBUKey"),
            lit(None).cast("string").alias("SayFlag"),
            lit(None).cast("string").alias("InsuranceCompany"),
            lit(None).cast("string").alias("PhysicalNumber"),
            lit(None).cast("string").alias("PreCourtDate"),
            lit(None).cast("string").alias("CourtDate"),
            lit(None).cast("string").alias("ContractSigningDate"),
            loan_details.ACC_LIMIT_AMN.cast("string").alias("ExposureLimitAmount"),
            loan_add_info.AMOUNT_DATA.cast("decimal(15,2)").alias("BaloonAmount"),
            lit(None).cast("integer").alias("AdministrationStatusCode"),
            lit(None).cast("integer").alias("AdministrationStatus"),
            lit(None).cast("string").alias("tenor"),
            lit(None).cast("boolean").alias("IsSoldFlag"),
            lit(None).cast("string").alias("IsSubsidyFlag"),
            lit(None).cast("boolean").alias("IsGuaranteeFlag"),
            lit(None).cast("string").alias("EverDenouncedFlag"),
            lit(None).cast("string").alias("EverDenouncedDate"),
            lit(None).cast("string").alias("SPVCode"),
            lit(None).cast("string").alias("SPV"),
            loan_details.ov_int_cnt.cast("decimal(4,0)").alias("TotalOverdueInsterest"),
            loan_details.ov_inst_cnt.cast("decimal(4,0)").alias(
                "TotalOverdueInstallments"
            ),
            lit(None).cast("decimal(19,4)").alias("OutstandingTotalBalanceMLT"),
            lit(None).alias("AllBorrowersDeceasedFlag"),
            lit(None).alias("TreatmentBranchCode"),
            lit(None).alias("TreatmentBranch"),
            lit(None).alias("LetterStatusCode"),
            lit(None).alias("LetterStatus"),
            lit(None).alias("LetterStatusDate"),
            lit(None).alias("CoCStatusCode"),
            lit(None).alias("CoCStatus"),
            lit(None).alias("CoCAccountStatusDate"),
            lit(None).alias("FirstLetterDate"),
            lit(None).alias("SecondLetterDate"),
            lit(None).alias("ThirdLetterDate"),
            lit(None).alias("FirstLetterReminderDate"),
            lit(None).alias("FirstLetterFlag"),
            lit(None).alias("FirstLetterReminderFlag"),
            lit(None).alias("SecondLetterFlag"),
            lit(None).alias("ThirdLetterFlag"),
            lit(None).alias("ExclusionDate"),
            lit(None).alias("ExclusionReasonCode"),
            lit(None).alias("ExclusionReason"),
            lit(None).alias("LegalActionPriority"),
            lit(None).alias("DueDebtAmount"),
            target_table_profits.JUSTIFICATION_CODE.cast("int").alias("JustificationCode"),
            target_table_profits.EXTRAIT_COMMENTS.cast("string").alias("JustificationComments"),
            loan_details.businessdate.alias("BusinessDate"),
            lit(None).cast("Boolean").alias("ForcedOnboarding"),
            lit(None).cast("Boolean").alias("HolisticFlag"),
            lit(None).cast("String").alias("ForcedOnboardReason"),  
            target_table_profits.URL_NRM_INT_AMN.cast("decimal(15,2)").alias("url_nrm_int_amn"),
        )
        
        #========================== Append ==========================
        result_df_profits.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")
    

In [0]:
load_curated_exposure_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)